# ST Score Restore — Stage 11 V2d Crash-Safe Colab Recovery

Development-only, inference-only recovery runner. Restore remains exploratory GPU work; Oemer detector inference is pinned to ONNX Runtime 1.20.1 CPUExecutionProvider. All reusable exact inputs, source/restored pages, and detector-page progress persist in Google Drive.

Colab Python 3.13 uses an isolated source-path recovery mode. Google Drive authentication and exact-input verification are completed in the notebook kernel before the benchmark subprocess starts.


In [ ]:
# 1) Pinned detector runtime for Colab Python 3.11-3.13.
# Keep NumPy below 2.3 because Colab's Numba 0.61.x requires numpy<2.3.
import sys, subprocess

subprocess.run(['nvidia-smi'], check=False)
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'poppler-utils'], check=True)

# Remove conflicting runtime variants before installing the single CPU ORT / headless OpenCV stack.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'onnxruntime', 'onnxruntime-gpu',
                'opencv-python', 'opencv-python-headless',
                'opencv-contrib-python', 'opencv-contrib-python-headless'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy>=2.0,<2.3',
                'onnxruntime==1.20.1',
                'opencv-python-headless==4.13.0.92',
                'scipy', 'scikit-learn', 'matplotlib', 'pillow', 'typing-extensions'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                'git+https://github.com/BreezeWhite/oemer@dbe2a933d630d0f74805d717960eb259473f5978'], check=True)

print('python', sys.version)
if not ((3, 11) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError('This Colab recovery notebook accepts Python 3.11-3.13 only.')
if sys.version_info[:2] == (3, 13):
    print('Python 3.13 Colab recovery mode: repository will be imported from source, not installed as a package.')

# Validate the exact runtime in a fresh interpreter so stale in-kernel imports cannot hide a bad install.
verify = r'''import sys
import numpy as np
import cv2
import onnxruntime as ort
import scipy
import sklearn
from importlib.util import find_spec
print('numpy', np.__version__)
print('opencv', cv2.__version__)
print('onnxruntime', ort.__version__, ort.get_available_providers())
print('scipy', scipy.__version__)
print('sklearn', sklearn.__version__)
major_minor = tuple(int(x) for x in np.__version__.split('.')[:2])
assert (2, 0) <= major_minor < (2, 3), np.__version__
assert ort.__version__ == '1.20.1'
assert 'CPUExecutionProvider' in ort.get_available_providers()
if find_spec('numba') is not None:
    import numba
    print('numba', numba.__version__)
import oemer
from oemer.ete import generate_pred
print('Oemer import PASS')
'''
subprocess.run([sys.executable, '-c', verify], check=True)
print('RUNTIME PREFLIGHT PASS')


In [ ]:
# 2) Mount Drive, authenticate in the real Colab kernel, refresh repo source, and prefill exact-input cache.
# IMPORTANT: google.colab.auth.authenticate_user() must not run inside the benchmark subprocess.
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

import google.auth
credentials, _ = google.auth.default()
if credentials is None:
    raise RuntimeError('Google Drive credentials unavailable after Colab authentication.')
print('DRIVE AUTH PREFLIGHT PASS')

import shutil, subprocess, sys
from pathlib import Path
REPO = Path('/content/st-score-restore-engine')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '-q', '-b', 'stage11-v2c-semantic-detector-corpus-expansion', 'https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
print('repo source ready', REPO)

# Materialize/verify all six exact Drive inputs in the notebook kernel.
# The later subprocess therefore has no reason to invoke Colab's kernel-only auth helper.
from st_score_restore.stage11_v2d_colab_runner import CACHE_ROOT, TARGETS, _download_exact_drive_targets
exact_inputs = _download_exact_drive_targets(CACHE_ROOT / 'exact_inputs')
if set(exact_inputs) != set(TARGETS):
    raise RuntimeError(f'Exact-input preflight incomplete: {sorted(exact_inputs)}')
print(f'EXACT INPUT PREFLIGHT PASS: {len(exact_inputs)}/{len(TARGETS)}')


In [ ]:
# 3) Run the benchmark in a fresh subprocess; exact Drive inputs are already verified and cached.
import os, subprocess, sys
from pathlib import Path
RESULT_DIR = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/V2D_RESULTS')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
LOG = RESULT_DIR / 'v2d_recovery_full.log'
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONFAULTHANDLER'] = '1'
env['PYTHONPATH'] = str(REPO / 'src') + os.pathsep + env.get('PYTHONPATH', '')
cmd = [sys.executable, '-m', 'st_score_restore.stage11_v2d_colab_runner', '--run']
print('running:', ' '.join(cmd))
print('log:', LOG)
with LOG.open('a', encoding='utf-8', buffering=1) as log_handle:
    log_handle.write('\n===== V2D RECOVERY RUN =====\n')
    proc = subprocess.Popen(
        cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        log_handle.write(line)
    returncode = proc.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, cmd)


## Başarı işaretleri

`RUNTIME PREFLIGHT PASS` → `DRIVE AUTH PREFLIGHT PASS` → `EXACT INPUT PREFLIGHT PASS: 6/6` → `Oemer preflight PASS` → `Oemer real-image smoke PASS: source + restored` → `V2D RECOVERY PASS` → `SAVED:`.

Aynı notebook yeniden çalıştırıldığında doğrulanmış Drive cache ve sayfa-bazlı detector JSON kayıtları yeniden kullanılır. Final canonical CPU rerun gereksinimi değişmez.
